# Notebook: Fine-tuning Thai Multilabel Emotion Detection (GoEmotions 28 Classes) - Full Train Set
**สถาปัตยกรรม:** Hybrid Stacked Ensemble (WangchanBERTa + mBERT + XLM-RoBERTa -> Random Forest)
**ชุดข้อมูล:**
- **Train Set:** `go_emotions_train_with_translation.csv` (43,410 แถว -> Full Train Set)
- **Validation Set:** `go_emotions_validation_with_translation.csv` (5,426 แถว -> Official Validation Set)
- **Test Set:** `go_emotions_test_with_translation.csv` (5,131 แถว -> Unseen Benchmark Test Set)
- **โมเดลที่บันทึก:** เก็บในโฟลเดอร์ `saved_models_full_train/` (ไม่ทับโมเดลเดิมใน `saved_models/`)


In [1]:
print("hello world")

hello world


## Step 1: โหลดไลบรารีและตั้งค่า Device (GPU/CPU)

In [2]:
import os
import ast
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.nn import BCEWithLogitsLoss
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import classification_report, hamming_loss, f1_score
from tqdm.notebook import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Import ฟังก์ชันบันทึกผลการประเมินลง Markdown (.md)
import sys, gc
if '..' not in sys.path:
    sys.path.append('..')
from src.eval_logger import save_eval_report


Device: cuda


## Step 2: โหลดข้อมูลภาษาไทย & Label Binarization (28 Classes)

In [3]:
TRAIN_PATH = "../goemotions/data/go_emotions_train_with_translation.csv"
if not os.path.exists(TRAIN_PATH):
    TRAIN_PATH = "goemotions/data/go_emotions_train_with_translation.csv"

VAL_PATH = "../goemotions/data/go_emotions_validation_with_translation.csv"
if not os.path.exists(VAL_PATH):
    VAL_PATH = "goemotions/data/go_emotions_validation_with_translation.csv"

TEST_PATH = "../goemotions/data/go_emotions_test_with_translation.csv"
if not os.path.exists(TEST_PATH):
    TEST_PATH = "goemotions/data/go_emotions_test_with_translation.csv"

# 1) โหลด Train Set (43,410 แถว), Val Set (5,426 แถว) และ Test Set (5,131 แถว)
df_train = pd.read_csv(TRAIN_PATH).dropna(subset=["text_th"]).reset_index(drop=True)
df_val = pd.read_csv(VAL_PATH).dropna(subset=["text_th"]).reset_index(drop=True)
df_test = pd.read_csv(TEST_PATH).dropna(subset=["text_th"]).reset_index(drop=True)
print(f"จำนวนข้อมูล Train ภาษาไทย      : {len(df_train):,} แถว")
print(f"จำนวนข้อมูล Validation ภาษาไทย : {len(df_val):,} แถว")
print(f"จำนวนข้อมูล Test ภาษาไทย       : {len(df_test):,} แถว")

EMOTION_LABELS = [
    "admiration", "amusement", "anger", "annoyance", "approval",
    "caring", "confusion", "curiosity", "desire", "disappointment",
    "disapproval", "disgust", "embarrassment", "excitement", "fear",
    "gratitude", "grief", "joy", "love", "nervousness",
    "optimism", "pride", "realization", "relief", "remorse",
    "sadness", "surprise", "neutral"
]
NUM_LABELS = len(EMOTION_LABELS)

df_train["labels_list"] = df_train["labels"].apply(ast.literal_eval)
df_val["labels_list"] = df_val["labels"].apply(ast.literal_eval)
df_test["labels_list"] = df_test["labels"].apply(ast.literal_eval)

mlb = MultiLabelBinarizer(classes=list(range(NUM_LABELS)))
Y_train = mlb.fit_transform(df_train["labels_list"])
X_train = df_train["text_th"].astype(str).tolist()

Y_val = mlb.transform(df_val["labels_list"])
X_val = df_val["text_th"].astype(str).tolist()

# X_test และ Y_test มาจาก Test Set ตัวจริงของ GoEmotions โดยตรง
Y_test = mlb.transform(df_test["labels_list"])
X_test = df_test["text_th"].astype(str).tolist()

# pos_weight สำหรับ Imbalanced Data คำนวณจาก Train Set
pos_counts = Y_train.sum(axis=0)
neg_counts = len(df_train) - pos_counts
pos_weight = neg_counts / (pos_counts + 1e-6)
pos_weight_tensor = torch.FloatTensor(pos_weight).to(device)

print(f"✅ สรุปขนาดข้อมูล: Train={len(X_train):,} (Official) | Val={len(X_val):,} (Official) | Test={len(X_test):,} (Official)")

จำนวนข้อมูล Train ภาษาไทย      : 43,410 แถว
จำนวนข้อมูล Validation ภาษาไทย : 5,426 แถว
จำนวนข้อมูล Test ภาษาไทย       : 5,131 แถว
✅ สรุปขนาดข้อมูล: Train=43,410 (Official) | Val=5,426 (Official) | Test=5,131 (Official)


## Step 3: PyTorch Dataset & Fine-tuning Base Models

In [4]:
class ThaiEmotionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        enc = self.tokenizer(text, max_length=self.max_len, padding="max_length", truncation=True, return_tensors="pt")
        return {
            "input_ids": enc["input_ids"].flatten(),
            "attention_mask": enc["attention_mask"].flatten(),
            "labels": torch.FloatTensor(self.labels[idx])
        }

def train_model(model_name, use_fast=True, epochs=3, lr=2e-5, batch_size=16):
    print(f"--- Fine-tuning {model_name} ---")
    tok = AutoTokenizer.from_pretrained(model_name, use_fast=use_fast)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=NUM_LABELS, problem_type="multi_label_classification").to(device)
    train_ds = ThaiEmotionDataset(X_train, Y_train, tok)
    val_ds = ThaiEmotionDataset(X_val, Y_val, tok)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    optimizer = AdamW(model.parameters(), lr=lr)
    criterion = BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for b in train_loader:
            optimizer.zero_grad()
            out = model(b["input_ids"].to(device), attention_mask=b["attention_mask"].to(device))
            loss = criterion(out.logits, b["labels"].to(device))
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for b in val_loader:
                out = model(b["input_ids"].to(device), attention_mask=b["attention_mask"].to(device))
                val_loss += criterion(out.logits, b["labels"].to(device)).item()
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {val_loss/len(val_loader):.4f}")
    # เคลียร์ memory ของ optimizer และ dataloader เพื่อประหยัด VRAM
    del optimizer, train_loader, val_loader
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    return model, tok

# เทรน 3 Base Models ด้วยข้อมูลชุดเต็ม
model_wangchan, tok_wangchan = train_model("airesearch/wangchanberta-base-att-spm-uncased", use_fast=False)
model_mbert, tok_mbert = train_model("bert-base-multilingual-cased", use_fast=True)
model_xlmr, tok_xlmr = train_model("xlm-roberta-base", use_fast=True)


--- Fine-tuning airesearch/wangchanberta-base-att-spm-uncased ---


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] CamembertForSequenceClassification LOAD REPORT from: airesearch/wangchanberta-base-att-spm-uncased
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/3 | Train Loss: 1.2689 | Val Loss: 1.3500
Epoch 2/3 | Train Loss: 1.3459 | Val Loss: 1.3398
Epoch 3/3 | Train Loss: 1.3425 | Val Loss: 1.3393
--- Fine-tuning bert-base-multilingual-cased ---


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/3 | Train Loss: 1.1442 | Val Loss: 1.0392
Epoch 2/3 | Train Loss: 0.9449 | Val Loss: 0.8942
Epoch 3/3 | Train Loss: 0.8433 | Val Loss: 0.8395
--- Fine-tuning xlm-roberta-base ---


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/3 | Train Loss: 1.3340 | Val Loss: 1.3359


KeyboardInterrupt: 

In [5]:
model_wangchan.save_pretrained("saved_models/wangchan_model")
tok_wangchan.save_pretrained("saved_models/wangchan_tokenizer")
model_mbert.save_pretrained("saved_models/mbert_model")
tok_mbert.save_pretrained("saved_models/mbert_tokenizer")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('saved_models/mbert_tokenizer/tokenizer_config.json',
 'saved_models/mbert_tokenizer/tokenizer.json')

In [6]:
model_wangchan.to("cpu")
model_mbert.to("cpu")
torch.cuda.empty_cache()


In [ ]:
import gc
import torch
from torch.amp import autocast, GradScaler
from torch.optim import AdamW
from torch.nn import BCEWithLogitsLoss
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm.auto import tqdm

# 1. ย้ายโมเดล 2 ตัวแรกที่เทรนเสร็จแล้วไปไว้ที่ CPU เพื่อคืน VRAM ให้ว่าง 100%
if 'model_wangchan' in globals():
    model_wangchan.to("cpu")
if 'model_mbert' in globals():
    model_mbert.to("cpu")
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

print("🧹 เคลียร์ VRAM เรียบร้อย พร้อมเทรนด้วย FP16 (Mixed Precision)!")

# 2. ฟังก์ชัน train_model ที่รองรับ FP16 + แสดงแถบ Progress Bar (tqdm)
def train_model_fp16(model_name, use_fast=True, epochs=3, lr=2e-5, batch_size=16):
    print(f"\n🚀 --- Fine-tuning {model_name} (FP16 Accelerated) ---")
    tok = AutoTokenizer.from_pretrained(model_name, use_fast=use_fast)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, 
        num_labels=NUM_LABELS, 
        problem_type="multi_label_classification"
    ).to(device)

    train_ds = ThaiEmotionDataset(X_train, Y_train, tok)
    val_ds   = ThaiEmotionDataset(X_val,   Y_val,   tok)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False)

    optimizer = AdamW(model.parameters(), lr=lr)
    criterion = BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
    
    # ตัวจัดการ Scaling สำหรับ FP16 ป้องกัน Underflow
    scaler = GradScaler('cuda' if torch.cuda.is_available() else 'cpu')

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        pbar_train = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
        
        for b in pbar_train:
            optimizer.zero_grad()
            input_ids = b["input_ids"].to(device)
            attention_mask = b["attention_mask"].to(device)
            labels = b["labels"].to(device)

            # รัน Forward Pass ด้วย FP16 อัตโนมัติ
            with autocast('cuda' if torch.cuda.is_available() else 'cpu'):
                out = model(input_ids, attention_mask=attention_mask)
                loss = criterion(out.logits, labels)

            # Backward pass ผ่าน GradScaler
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item()
            pbar_train.set_postfix({"loss": f"{loss.item():.4f}"})

        # Validation Loop
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            pbar_val = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]")
            for b in pbar_val:
                input_ids = b["input_ids"].to(device)
                attention_mask = b["attention_mask"].to(device)
                labels = b["labels"].to(device)

                with autocast('cuda' if torch.cuda.is_available() else 'cpu'):
                    out = model(input_ids, attention_mask=attention_mask)
                    val_loss += criterion(out.logits, labels).item()

        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        print(f"✨ Epoch {epoch+1}/{epochs} -> Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    del optimizer, train_loader, val_loader, scaler
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    return model, tok

# 3. สั่งเทรนเฉพาะ XLM-RoBERTa ตัวเดียว (ไม่ต้องเทรน 2 ตัวแรกซ้ำ!)
model_xlmr, tok_xlmr = train_model_fp16("xlm-roberta-base", use_fast=True, epochs=3, batch_size=16)



🧹 เคลียร์ VRAM เรียบร้อย พร้อมเทรนด้วย FP16 (Mixed Precision)!

🚀 --- Fine-tuning xlm-roberta-base (FP16 Accelerated) ---


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/3 [Train]:   0%|          | 0/2714 [00:00<?, ?it/s]

Epoch 1/3 [Val]:   0%|          | 0/340 [00:00<?, ?it/s]

✨ Epoch 1/3 -> Train Loss: 1.3335 | Val Loss: 1.3357


Epoch 2/3 [Train]:   0%|          | 0/2714 [00:00<?, ?it/s]

## Step 4: สกัด 84-dim Probabilities & เทรน Random Forest Meta-Classifier

In [ ]:
def get_probs(model, tok, texts):
    model.eval()
    ds = ThaiEmotionDataset(texts, np.zeros((len(texts), NUM_LABELS)), tok)
    loader = DataLoader(ds, batch_size=32, shuffle=False)
    probs = []
    with torch.no_grad():
        for b in loader:
            out = model(b["input_ids"].to(device), attention_mask=b["attention_mask"].to(device))
            probs.append(torch.sigmoid(out.logits).cpu().numpy())
    return np.concatenate(probs, axis=0)

print("ดึง Probability Features...")
train_w = get_probs(model_wangchan, tok_wangchan, X_train)
train_m = get_probs(model_mbert, tok_mbert, X_train)
train_x = get_probs(model_xlmr, tok_xlmr, X_train)
X_train_meta = np.concatenate([train_w, train_m, train_x], axis=1)

test_w = get_probs(model_wangchan, tok_wangchan, X_test)
test_m = get_probs(model_mbert, tok_mbert, X_test)
test_x = get_probs(model_xlmr, tok_xlmr, X_test)
X_test_meta = np.concatenate([test_w, test_m, test_x], axis=1)

rf = MultiOutputClassifier(RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42, n_jobs=-1))
rf.fit(X_train_meta, Y_train)
print("✅ เทรน Random Forest Meta-Classifier สำเร็จ!")

## Step 5: ประเมินผลลัพธ์ (Evaluation)

In [ ]:
Y_pred = rf.predict(X_test_meta)
print(f"Micro F1: {f1_score(Y_test, Y_pred, average='micro'):.4f}")
print(f"Macro F1: {f1_score(Y_test, Y_pred, average='macro'):.4f}")
print(f"Hamming Loss: {hamming_loss(Y_test, Y_pred):.4f}")
print("Classification Report:")
print(classification_report(Y_test, Y_pred, target_names=EMOTION_LABELS, zero_division=0))

# บันทึกผลการประเมินลงไฟล์ Markdown (.md) ในโฟลเดอร์ results/ พร้อมอัปเดต SUMMARY.md
report_path = save_eval_report(
    y_true=Y_test,
    y_pred=Y_pred,
    target_names=EMOTION_LABELS,
    model_name="Stacked Ensemble Full Train (WangchanBERTa + mBERT + XLM-RoBERTa -> Random Forest)",
    hyperparams={
        "Dataset": "Train 43,410 rows -> Test Official 5,131 rows",
        "Base Models": "WangchanBERTa, mBERT, XLM-RoBERTa",
        "Meta Classifier": "RandomForest (n_estimators=200)",
        "Epochs": 3,
        "Batch Size": 16,
        "Learning Rate": "2e-5"
    },
    notes="การประเมินจาก Notebook 04_base_models_full_train.ipynb (Full Train Set 43k)",
    output_dir="../results" if os.path.exists("../results") else "results"
)


## Step 6: บันทึกโมเดล (Save Models & Artifacts)

In [ ]:
# บันทึกโมเดลลงโฟลเดอร์ saved_models_full_train เพื่อป้องกันไม่ให้ทับโฟลเดอร์ saved_models เดิม
SAVE_DIR = "saved_models_full_train"
if not os.path.exists(SAVE_DIR) and os.path.exists("../saved_models"):
    SAVE_DIR = "../saved_models_full_train"
os.makedirs(SAVE_DIR, exist_ok=True)

model_wangchan.save_pretrained(f"{SAVE_DIR}/wangchan_model")
tok_wangchan.save_pretrained(f"{SAVE_DIR}/wangchan_tokenizer")
model_mbert.save_pretrained(f"{SAVE_DIR}/mbert_model")
tok_mbert.save_pretrained(f"{SAVE_DIR}/mbert_tokenizer")
model_xlmr.save_pretrained(f"{SAVE_DIR}/xlm_roberta_model")
tok_xlmr.save_pretrained(f"{SAVE_DIR}/xlm_roberta_tokenizer")

joblib.dump(rf, f"{SAVE_DIR}/random_forest_meta.joblib")
with open(f"{SAVE_DIR}/label_classes.json", "w", encoding="utf-8") as f:
    json.dump(EMOTION_LABELS, f, ensure_ascii=False, indent=2)
print(f"✅ บันทึกโมเดลชุดใหม่ทั้งหมดลงใน {SAVE_DIR} เรียบร้อย! (ไม่ทับ saved_models เดิม)")

## Step 7: เคลียร์โมเดลและคืน VRAM GPU (Release GPU Memory)

In [ ]:
# ==========================================
# Step 7: เคลียร์โมเดลและคืน VRAM GPU ทั้งหมด
# ==========================================
import gc
print('กำลังเคลียร์โมเดลออกจาก VRAM...')
if 'model_wangchan' in locals(): del model_wangchan
if 'model_mbert' in locals(): del model_mbert
if 'model_xlmr' in locals(): del model_xlmr

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
gc.collect()
print('✅ คืน VRAM GPU สำเร็จ!')
if torch.cuda.is_available():
    print(f'VRAM คงเหลือที่ถูกใช้งาน: {torch.cuda.memory_allocated() / 1e6:.2f} MB')
